# Kvantu reģistru saskaitīšana un atņemšana Qiskit vidē

Šajā notebook tiek parādītas divas metodes, kā saskaitīt un atņemt divus $n$-kubit reģistrus $a$ un $b$ tā, ka rezultāts tiek ierakstīts reģistrā $b$:

- **Ripple-carry (Cuccaro MAJ/UMA):** izmanto pārnesumu $c$ (1 palīgkubits). 
- **Draper QFT adder:** izmanto QFT un kontrolētas fāžu rotācijas, bez palīgkubitiem.


# SASKAITĪŠANA

## 1) Ripple-carry saskaitītājs

Ideja ir tāda pati kā klasiskajā binārajā saskaitīšanā. Ejot no LSB uz MSB, tiek nests pārnesums (carry) palīgkubītā $c$, kas sākumā ir $|0\rangle$.

Cuccaro adderis atkārto divus blokus katram bitam $i$:

- **MAJ($a_i,b_i,c$)** - aprēķina nākamo pārnesumu $c_{i+1}$ (majority no $a_i,b_i,c$) un padod, jeb “ripplē” to uz priekšu.  
- **UMA($a_i,b_i,c$)** - iet atpakaļ, atjauno $a_i$ un atstāj summas bitu reģistrā $b$.

### MAJ($a_i,b_i,c$) vārti
1) $CX(c \rightarrow b_i)$  
2) $CX(c \rightarrow a_i)$
4) $CCX(a_i, b_i \rightarrow c)$  

### UMA($a_i,b_i,c$) vārti
1) $CCX(a_i, b_i \rightarrow c)$  
2) $CX(c \rightarrow a_i)$  
3) $CX(a_i \rightarrow b_i)$  

### Pilna shēma ar $n$ bitiem
1) **Padošana uz priekšu (LSB uz MSB)**: MAJ($a_0,b_0,c$), …, MAJ($a_{n-1},b_{n-1},c$)  
2) **Padošana atpakaļ (MSB uz LSB)**: UMA($a_{n-1},b_{n-1},c$), …, UMA($a_0,b_0,c$)

Rezultāts: $b$ satur $(a+b)\bmod 2^n$, $a$ paliek kā bija, bet $c$ satur gala pārnesumu (carry-out).

In [1]:
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister, transpile
from qiskit_aer import AerSimulator

sim = AerSimulator()

def maj(qc, a, b, c):
    qc.cx(c, b)
    qc.cx(c, a)
    qc.ccx(a, b, c)

def uma(qc, a, b, c):
    qc.ccx(a, b, c)
    qc.cx(c, a)
    qc.cx(a, b)

def ripple_add(qc, a, b, carry):
    n = len(a)
    # carry uz priekšu
    for i in range(n):
        maj(qc, a[i], b[i], carry)
    # carry atpakaļ + summa b reģistrā
    for i in reversed(range(n)):
        uma(qc, a[i], b[i], carry)

# -------------------------------------------------
# Piemērs 1 
a1 = QuantumRegister(3, "a")
b1 = QuantumRegister(3, "b")
carry1 = QuantumRegister(1, "carry")
cb1 = ClassicalRegister(3, "cb")
qc1 = QuantumCircuit(a1, b1, carry1, cb1)

# a = 5 = 101
qc1.x(a1[0])
qc1.x(a1[2])

# b = 3 = 011
qc1.x(b1[0])
qc1.x(b1[1])

ripple_add(qc1, a1, b1, carry1[0])
qc1.measure(b1, cb1)

tqc1 = transpile(qc1, sim)
counts1 = sim.run(tqc1, shots=1024).result().get_counts()

print("Ripple-carry: a=5, b=3")
print("Summa: ", counts1)
print(qc1.draw("text"))

# -------------------------------------------------
# Piemērs 2 (superpozīcija): 
a2 = QuantumRegister(3, "a")
b2 = QuantumRegister(3, "b")
carry2 = QuantumRegister(1, "carry")
cb2 = ClassicalRegister(3, "cb")
qc2 = QuantumCircuit(a2, b2, carry2, cb2)

# a = (|000>+|100>)/sqrt2
qc2.h(a2[2])

# b = 1 = 001
qc2.x(b2[0])

ripple_add(qc2, a2, b2, carry2[0])

qc2.measure(b2, cb2)

tqc2 = transpile(qc2, sim)
counts2 = sim.run(tqc2, shots=4096).result().get_counts()

print("Ripple-carry: a=(|000>+|100>)/sqrt2, b=1 ")
print("Summa: ", counts2)
print(qc2.draw("text"))


Ripple-carry: a=5, b=3
Summa:  {'000': 1024}
       ┌───┐     ┌───┐                                                       »
  a_0: ┤ X ├─────┤ X ├──■────────────────────────────────────────────────────»
       └───┘     └─┬─┘  │       ┌───┐                                        »
  a_1: ────────────┼────┼───────┤ X ├──■──────────────────────────────────■──»
       ┌───┐       │    │       └─┬─┘  │       ┌───┐          ┌───┐       │  »
  a_2: ┤ X ├───────┼────┼─────────┼────┼───────┤ X ├──■────■──┤ X ├──■────┼──»
       ├───┤┌───┐  │    │         │    │       └─┬─┘  │    │  └─┬─┘  │    │  »
  b_0: ┤ X ├┤ X ├──┼────■─────────┼────┼─────────┼────┼────┼────┼────┼────┼──»
       ├───┤└─┬─┘  │    │  ┌───┐  │    │         │    │    │    │    │    │  »
  b_1: ┤ X ├──┼────┼────┼──┤ X ├──┼────■─────────┼────┼────┼────┼────┼────■──»
       └───┘  │    │    │  └─┬─┘  │    │  ┌───┐  │    │    │    │  ┌─┴─┐  │  »
  b_2: ───────┼────┼────┼────┼────┼────┼──┤ X ├──┼────■────■────┼──┤ X ├──┼──»
       

## 2) Draper QFT saskaitītājs

Šī metode izmanto Kvantu Furjē transformāciju (QFT) uz reģistra $b$.

QFT pārnes $|b\rangle$ fāžu telpā, kur pieskaitīšana var tikt realizēta ar kontrolētām fāžu rotācijām atkarībā no $a$ bitiem. Pēc tam $\mathrm{QFT}^\dagger$ atgriež $b$ atpakaļ skaitļa formā.

Mērķis ir tāds pats:

$b \leftarrow (a+b)\bmod 2^n,$ un $a$ paliek nemainīgs.

### Augsta līmeņa soļi
1) $\mathrm{QFT}$ uz $b$  
2) kontrolētas rotācijas starp $a$ un $b$  
3) $\mathrm{QFT}^\dagger$ uz $b$

Python to realizē ar **DraperQFTAdder(n, kind="fixed")**

In [2]:
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister, transpile
from qiskit.circuit.library import DraperQFTAdder
from qiskit_aer import AerSimulator

sim = AerSimulator()
adder = DraperQFTAdder(3, kind="fixed")

# -------------------------------------------------
# Piemērs 1:
a1 = QuantumRegister(3, "a")
b1 = QuantumRegister(3, "b")
cb1 = ClassicalRegister(3, "cb")
qc1 = QuantumCircuit(a1, b1, cb1)

# a = 5 = 101 
qc1.x(a1[0])
qc1.x(a1[2])

# b = 3 = 011
qc1.x(b1[0])
qc1.x(b1[1])

qc1.append(adder, [*a1, *b1])
qc1.measure(b1, cb1)

tqc1 = transpile(qc1, sim)
counts1 = sim.run(tqc1, shots=1024).result().get_counts()

print("Draper QFT: a=5 (101), b=3 (011)")
print("Rezultāts:", counts1)
print(qc1.draw("text"))

# -------------------------------------------------
# Piemērs 2:

a2 = QuantumRegister(3, "a")
b2 = QuantumRegister(3, "b")
cb2 = ClassicalRegister(3, "cb")
qc2 = QuantumCircuit(a2, b2, cb2)

# a = 0 vai 4 ((|000>+|100>)/sqrt2)
qc2.h(a2[2])

# b=1 (001)
qc2.x(b2[0])

qc2.append(adder, [*a2, *b2])
qc2.measure(b2, cb2)

tqc2 = transpile(qc2, sim)
counts2 = sim.run(tqc2, shots=4096).result().get_counts()

print("Draper QFT: a=(|000>+|100>)/sqrt2, b=1 (001)")
print("Rezultāts:", counts2)
print(qc2.draw("text"))


Draper QFT: a=5 (101), b=3 (011)
Rezultāts: {'000': 1024}
      ┌───┐┌─────────────────┐         
 a_0: ┤ X ├┤0                ├─────────
      └───┘│                 │         
 a_1: ─────┤1                ├─────────
      ┌───┐│                 │         
 a_2: ┤ X ├┤2                ├─────────
      ├───┤│  DraperQFTAdder │┌─┐      
 b_0: ┤ X ├┤3                ├┤M├──────
      ├───┤│                 │└╥┘┌─┐   
 b_1: ┤ X ├┤4                ├─╫─┤M├───
      └───┘│                 │ ║ └╥┘┌─┐
 b_2: ─────┤5                ├─╫──╫─┤M├
           └─────────────────┘ ║  ║ └╥┘
cb: 3/═════════════════════════╩══╩══╩═
                               0  1  2 
Draper QFT: a=(|000>+|100>)/sqrt2, b=1 (001)
Rezultāts: {'101': 1960, '001': 2136}
           ┌─────────────────┐         
 a_0: ─────┤0                ├─────────
           │                 │         
 a_1: ─────┤1                ├─────────
      ┌───┐│                 │         
 a_2: ┤ H ├┤2                ├─────────
      ├───┤│  Drape

# ATŅEMŠANA

## 1) Ripple-carry atņemšana

Atņemšana $b \leftarrow (b-a)\bmod 2^n$ tiek iegūta, izpildot to pašu shēmu ko saskaitīšanā, tikai pretējā virzienā, ņemot inverso.

Cuccaro gadījumā visi vārti ir $\mathrm{CX}$ un $\mathrm{CCX}$. To inversi ir tie paši:
- $\mathrm{CX}^{-1}=\mathrm{CX}$
- $\mathrm{CCX}^{-1}=\mathrm{CCX}$
  
Tāpēc apgriezt nozīmē, apgriezt tikai secību un fāzes nav jāmaina.

### MAJ un UMA apgriezšana

Ja saskaitīšanā:

**MAJ($a_i,b_i,c$)** ir:
1) $\mathrm{CX}(c \rightarrow b_i)$  
2) $\mathrm{CX}(c \rightarrow a_i)$  
3) $\mathrm{CCX}(a_i, b_i \rightarrow c)$

**UMA($a_i,b_i,c$)** ir:
1) $\mathrm{CCX}(a_i, b_i \rightarrow c)$  
2) $\mathrm{CX}(c \rightarrow a_i)$  
3) $\mathrm{CX}(a_i \rightarrow b_i)$  

Tad atņemšanā:

**MAJ$^{-1}$** ir:
1) $\mathrm{CCX}(a_i, b_i \rightarrow c)$  
2) $\mathrm{CX}(c \rightarrow a_i)$  
3) $\mathrm{CX}(c \rightarrow b_i)$  

**UMA$^{-1}$** ir:
1) $\mathrm{CX}(a_i \rightarrow b_i)$  
2) $\mathrm{CX}(c \rightarrow a_i)$  
3) $\mathrm{CCX}(a_i, b_i \rightarrow c)$  


### Atņemšanas izpildes shēma

1) izpilda **UMA$^{-1}$** no LSB uz MSB  
   UMA$^{-1}$($a_0,b_0,c$), …, UMA$^{-1}$($a_{n-1},b_{n-1},c$)
2) izpilda **MAJ$^{-1}$** no MSB uz LSB  
   MAJ$^{-1}$($a_{n-1},b_{n-1},c$), …, MAJ$^{-1}$($a_0,b_0,c$)


In [3]:
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister, transpile
from qiskit_aer import AerSimulator

sim = AerSimulator()

def maj_inv(qc, a, b, c):
    qc.ccx(a, b, c)
    qc.cx(c, a)
    qc.cx(c, b)

def uma_inv(qc, a, b, c):
    qc.cx(a, b)
    qc.cx(c, a)
    qc.ccx(a, b, c)

def ripple_sub(qc, a, b, carry):
    n = len(a)
    for i in range(n):
        uma_inv(qc, a[i], b[i], carry)
    for i in reversed(range(n)):
        maj_inv(qc, a[i], b[i], carry)

# -------------------------------------------------
# Piemērs
a = QuantumRegister(3, "a")
b = QuantumRegister(3, "b")
carry = QuantumRegister(1, "carry")
cb = ClassicalRegister(3, "cb")
qc = QuantumCircuit(a, b, carry, cb)

# a = 5(101)
qc.x(a[0]); qc.x(a[1])
# b = 3(011)
qc.x(b[0]); qc.x(b[2])

ripple_sub(qc, a, b, carry[0])  

qc.measure(b, cb)

counts = sim.run(transpile(qc, sim), shots=1024).result().get_counts()
print("Ripple-carry atņemšana: a=3 (101), b=5 (011)")
print("Rezultāts:", counts)
print(qc.draw("text"))

Ripple-carry atņemšana: a=3 (101), b=5 (011)
Rezultāts: {'010': 1024}
       ┌───┐               ┌───┐                                             »
  a_0: ┤ X ├──■────────────┤ X ├──■──────────────────────────────────────────»
       ├───┤  │            └─┬─┘  │  ┌───┐                                   »
  a_1: ┤ X ├──┼────■─────────┼────┼──┤ X ├──■─────────────────────────────■──»
       └───┘  │    │         │    │  └─┬─┘  │  ┌───┐          ┌───┐       │  »
  a_2: ───────┼────┼────■────┼────┼────┼────┼──┤ X ├──■────■──┤ X ├───────┼──»
       ┌───┐┌─┴─┐  │    │    │    │    │    │  └─┬─┘  │    │  └─┬─┘       │  »
  b_0: ┤ X ├┤ X ├──┼────┼────┼────■────┼────┼────┼────┼────┼────┼─────────┼──»
       └───┘└───┘┌─┴─┐  │    │    │    │    │    │    │    │    │         │  »
  b_1: ──────────┤ X ├──┼────┼────┼────┼────■────┼────┼────┼────┼─────────■──»
       ┌───┐     └───┘┌─┴─┐  │    │    │    │    │    │    │    │  ┌───┐  │  »
  b_2: ┤ X ├──────────┤ X ├──┼────┼────┼────┼────┼────■────■─

## 2) DraperQFT atņemšana

Atņemšana ar Draper saskaitītāju arī tiek iegūta, ņemot inverso no saskaitīšanas shēmas.

Atšķirība no ripple-carry, Draper saskaitītājs sastāv no QFT un kontrolētām fāžu rotācijām, tāpēc apgriežot ir jāapgriež arī rotāciju virziens (leņķa zīme).

Draper atņemšanas inversā shēma:
1) $\mathrm{QFT}$ uz $b$
2) Tās pašas kontrolētās rotācijas, bet ar pretēju leņķa zīmi (piem., $R_k(\theta) \rightarrow R_k(-\theta)$)
3) $\mathrm{QFT}^\dagger$ uz $b$



In [4]:
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister, transpile
from qiskit.circuit.library import DraperQFTAdder
from qiskit_aer import AerSimulator

sim = AerSimulator()

adder = DraperQFTAdder(3, kind="fixed")

# Atņemšana ar inverse
sub = adder.inverse()

# -------------------------------------------------
# Piemērs: 
a = QuantumRegister(3, "a")
b = QuantumRegister(3, "b")
cb = ClassicalRegister(3, "cb")
qc = QuantumCircuit(a, b, cb)

# a = 5 = 101
qc.x(a[0]) 
qc.x(a[2])
# b = 3 = 011
qc.x(b[0]) 
qc.x(b[1])

qc.append(sub, [*a, *b])    
qc.measure(b, cb)

counts1 = sim.run(transpile(qc1, sim), shots=1024).result().get_counts()
print("Draper QFT atņemšana: a=5, b=3")
print("Rezultāts:", counts1)
print(qc1.draw("text"))



Draper QFT atņemšana: a=5, b=3
Rezultāts: {'000': 1024}
      ┌───┐┌─────────────────┐         
 a_0: ┤ X ├┤0                ├─────────
      └───┘│                 │         
 a_1: ─────┤1                ├─────────
      ┌───┐│                 │         
 a_2: ┤ X ├┤2                ├─────────
      ├───┤│  DraperQFTAdder │┌─┐      
 b_0: ┤ X ├┤3                ├┤M├──────
      ├───┤│                 │└╥┘┌─┐   
 b_1: ┤ X ├┤4                ├─╫─┤M├───
      └───┘│                 │ ║ └╥┘┌─┐
 b_2: ─────┤5                ├─╫──╫─┤M├
           └─────────────────┘ ║  ║ └╥┘
cb: 3/═════════════════════════╩══╩══╩═
                               0  1  2 
